[Guide Index](README.md)

# 05.11 SHACL `evaluate()`: shaping data for another process

`evaluate()` uses a shape to define and refine the shape of the data that another process will use, such as a report or a user interface. It does two things:

- **It selects what to present.** The result holds the nodes that the shapes target and, for each of them, the values of the properties that the shapes declare. Everything else in the data graph is left out.
- **It calculates values and assigns defaults.** A property declared with `sh:values` gets a computed value, and one declared with `sh:defaultValue` gets a default when it has no stored value.

The result is a new evaluation graph. The data graph is never changed, and the computed values and defaults are never stored. In SHACL Core, `sh:values` and `sh:defaultValue` can only give constant values, such as an IRI or literal as in the examples below. Computing a value from the data, for example from a property path or a count, needs node expressions. More complex uses of `evaluate()` and `sh:values` are shown in the [node expressions guide](05.04-shacl-node-expressions.ipynb).

By default `evaluate()` computes the values for every node a shape applies to and does not validate. Pass `validate=True` to evaluate only those nodes that conform to the shape that targets them. A node that does not conform contributes nothing to the evaluate graph. `evaluate()` returns only the graph and no report, so the report from running `validate()` on the same data is what says why a node was left out.  Typically a data graph would be validated prior to calling evaluate so any validation error can be fixed. 

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01.01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.

In [1]:
from starlayer.graph import Namespace
from starlayer.graph import StarLayerGraph
from starlayer.graph.config import set_default_prefixes
from starlayer.shacl import StarShaclProcessor

# EX and SH are starlayer.graph Namespaces: their terms are pyoxigraph NamedNodes, used to look terms up in the results below
# (StarLayerGraphs, see section 05.01.2); the StarLayerGraph inputs are built by parsing Turtle.
EX = Namespace("http://example.org/")
SH = Namespace("http://www.w3.org/ns/shacl#")

# Every StarLayerGraph made after this has these prefixes, so results, queries and the report text use short names such as
# ex:alice even for a graph built in Python rather than parsed from Turtle (a prefix the Turtle declares itself still applies).
set_default_prefixes({
    "ex": "http://example.org/",
    "sh": "http://www.w3.org/ns/shacl#",
    "shnex": "http://www.w3.org/ns/shacl-node-expr#",
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "xsd": "http://www.w3.org/2001/XMLSchema#",
})


def show(graph):
    """Print a graph as pretty Turtle, declaring only the prefixes it uses."""
    print(graph.serialize(format="turtle", pretty=True, only_used=True))

## 05.11.1 Selecting nodes and values, and computing a value

Below, the shape targets the people, declares `ex:name` so that it is presented, and computes `ex:valid` as the constant `true`. The result holds the people and those two properties, and nothing else.

In [2]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:name "Alice" ; ex:age 30 ; ex:ssn "123-45-6789" .
    ex:bob a ex:Person ; ex:name "Bob" ; ex:age 41 .
    ex:acme a ex:Company ; ex:name "Acme" .
""", format="turtle")

# the shape selects the people, declares ex:name to present, and adds ex:valid with the constant true
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .

    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ] ;
      sh:property [ sh:path ex:valid ; sh:values true ] .
""", format="turtle")

# the processor is created from the shapes graph, then used to evaluate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)
display_result = myProcessor.evaluate(data_graph=data)
print("nodes in the result:", sorted(display_result.qname(n) for n in set(display_result.subjects())))
print("triples in the data graph:", len(data), "| triples in the result:", len(display_result))

# the result holds only the people, their declared ex:name and the computed ex:valid:
# no ages, no ssn, no types, and not the company
show(display_result)


nodes in the result: ['ex:alice', 'ex:bob']
triples in the data graph: 9 | triples in the result: 4
@prefix ex: <http://example.org/> .

ex:alice ex:name "Alice" ;
    ex:valid true .

ex:bob ex:name "Bob" ;
    ex:valid true .



## 05.11.2 Stored values, computed values and defaults

A property computed with `sh:values` is normally one that is not expected to exist in the graph. If a stored value does exist, `evaluate()` returns it as well as the computed one. This is often not the desired behavior.

A property declared with `sh:defaultValue` supplies a value only when the node has no stored or computed value for it. A stored or computed value always wins, and the default is never used when another value is available.  The default value is present in the evaluation graph but not the data graph.

Below, `ex:alice` has a stored `ex:valid` of `false` and no status, and the shape computes `ex:valid` as `true` and defaults `ex:status` to `"unknown"`.

In [3]:
conflicting_data = StarLayerGraph()
conflicting_data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:name "Alice" ; ex:valid false .
   
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .

    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:property [ sh:path ex:valid ; sh:values true ] ;
      sh:property [ sh:path ex:status ; sh:defaultValue "unknown" ] .
""", format="turtle")

# the processor is created from the shapes graph, then used to evaluate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)
display_result = myProcessor.evaluate(data_graph=conflicting_data)


# the display graph holds the computed values and defaults, with any stored values of the same properties
show(display_result)


@prefix ex: <http://example.org/> .

ex:alice ex:status "unknown" ;
    ex:valid false, true .



## Related guides

- [Validating with SHACL 1.2](05.01-validating-with-shacl-1.2.ipynb) — `validate()`, which reports why a node does not conform.
- [SHACL node expressions](05.04-shacl-node-expressions.ipynb) — computing values from the data with `shnex:` and `sparql:` expressions, shown with `evaluate()`.
- [The StarLayer SHACL engine](05.02-starlayer-shacl-engine.ipynb) — configuring the processor, and the other methods and functions of the engine.